# CE, RobustCE, and label smoothing: fixed loss variants

## 1. Protocol

Compare logistic regression on overlapping Gaussian classes and a 2-8-1 tanh network on noisy semicircles. All computations use PyTorch on CPU.

Compare five fixed variants: **CE**, **RobustCE(0.05)**, **RobustCE(0.1)**, **RobustCE(0.2)**, and **smoothing with epsilon = 0.2**. Tune **only the learning rate**, separately for each variant and dataset, using training labels corrupted at **eta = 0.1** and ordinary CE on clean validation labels over three tuning seeds. The three RobustCE radii are separate comparison variants, not a radius-selection search.

Freeze the selected learning rates. Then train fresh models at **eta = 0** and **eta = 0.1** over five new evaluation seeds.

Within each replicate, variants share the data, initial parameters, observed labels, optimizer family, full-batch size, and epoch budget. Validation and test labels remain clean. Test results never influence learning-rate selection.

All figures use Plotly. Detailed records are saved to files without displaying tables. This notebook runs on its own and includes the radius/confidence comparison for critical analysis (b).


In [1]:
import csv
import json
import math
import statistics
import time
from itertools import combinations, product
from pathlib import Path

import plotly.graph_objects as go
import plotly.io as pio
import torch
import torch.nn as nn
import torch.nn.functional as F
from plotly.subplots import make_subplots

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "pyproject.toml").exists()
RESULTS_DIR = PROJECT_ROOT / "results" / "02_fixed_losses"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cpu")
DTYPE = torch.float32
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
pio.templates.default = "plotly_white"
pio.renderers.default = "plotly_mimetype"

DATASETS = ("gaussian", "moons")
EPSILON = 0.2
METHOD_CONFIGS = {
    "CE": {"delta": None, "epsilon": None},
    "RobustCE(0.05)": {"delta": 0.05, "epsilon": None},
    "RobustCE(0.1)": {"delta": 0.1, "epsilon": None},
    "RobustCE(0.2)": {"delta": 0.2, "epsilon": None},
    "Smoothing(0.2)": {"delta": None, "epsilon": EPSILON},
}
METHODS = tuple(METHOD_CONFIGS)
LEARNING_RATES = (0.001, 0.01, 0.05)
TUNING_CORRUPTION_RATE = 0.1
CORRUPTION_RATES = (0.0, 0.1)
TUNING_SEEDS = (0, 1, 2)
EVALUATION_SEEDS = (101, 102, 103, 104, 105)
SPLIT_SIZES = {"train": 1000, "validation": 400, "test": 2000}
EPOCHS = {"gaussian": 400, "moons": 800}
HISTORY_EVERY = 20
DELTA_EVAL = 0.1
REPRESENTATIVE_SEED = EVALUATION_SEEDS[0]
PLOT_CORRUPTION_RATES = CORRUPTION_RATES
COLORS = {
    "CE": "#2563eb", "RobustCE(0.05)": "#0891b2", "RobustCE(0.1)": "#ea580c",
    "RobustCE(0.2)": "#7c3aed", "Smoothing(0.2)": "#16a34a",
}
LINE_DASHES = {
    "CE": "solid", "RobustCE(0.05)": "longdash", "RobustCE(0.1)": "dash",
    "RobustCE(0.2)": "dashdot", "Smoothing(0.2)": "dot",
}

assert set(TUNING_SEEDS).isdisjoint(EVALUATION_SEEDS)
assert all(size % 2 == 0 for size in SPLIT_SIZES.values())
assert 0 < EPSILON <= 1
assert math.isclose(EPSILON / 2, METHOD_CONFIGS["RobustCE(0.1)"]["delta"])
print(f"PyTorch {torch.__version__} | CPU | {DTYPE}")
print("Variants: " + ", ".join(METHODS) + f"; smoothing epsilon={EPSILON}.")
print(f"Tune lr at eta={TUNING_CORRUPTION_RATE}; evaluate at eta={CORRUPTION_RATES}.")
print(f"Tuning seeds: {TUNING_SEEDS}; evaluation seeds: {EVALUATION_SEEDS}.")


PyTorch 2.11.0+cpu | CPU | torch.float32
Variants: CE, RobustCE(0.05), RobustCE(0.1), RobustCE(0.2), Smoothing(0.2); smoothing epsilon=0.2.
Tune lr at eta=0.1; evaluate at eta=(0.0, 0.1).
Tuning seeds: (0, 1, 2); evaluation seeds: (101, 102, 103, 104, 105).


## 2. Data, models, and shared randomness

Gaussian classes: $x\mid y\sim\mathcal N((2y-1)(1,1)^\top,I_2)$.

Semicircles: sample $t\sim U[0,\pi]$ and use $(\cos t,\sin t)$ for class 0 and $(1-\cos t,0.5-\sin t)$ for class 1. Add Gaussian noise with covariance $0.12I_2$, hence standard deviation $\sqrt{0.12}$.

Each split has exactly balanced clean labels and is generated independently. Standardization uses only training features. Seeds vary datasets and initialization. Within a seed, methods and learning rates receive the same data and initial parameters.

For each dataset and seed, draw one uniform vector and flip training label $i$ iff $u_i<\eta$. Share this vector across methods, learning rates, and evaluation conditions. The realized fraction of flipped labels can differ from the flip probability $\eta$.

Logistic regression has 3 parameters. The two-layer tanh network has 33. Fixed budgets are 400 and 800 full-batch Adam steps, respectively.


In [2]:
def generator(seed):
    return torch.Generator(device="cpu").manual_seed(seed)


def raw_split(dataset, n, seed):
    g = generator(seed)
    y = torch.cat((torch.zeros(n // 2), torch.ones(n // 2))).to(DTYPE)
    if dataset == "gaussian":
        x = torch.randn(n, 2, generator=g, dtype=DTYPE)
        x = x + (2 * y - 1).unsqueeze(1)
    elif dataset == "moons":
        t = torch.rand(n, generator=g, dtype=DTYPE) * math.pi
        x0 = torch.stack((t.cos(), t.sin()), dim=1)
        x1 = torch.stack((1 - t.cos(), 0.5 - t.sin()), dim=1)
        x = torch.where(y[:, None] == 0, x0, x1)
        x += math.sqrt(0.12) * torch.randn(n, 2, generator=g, dtype=DTYPE)
    else:
        raise ValueError(dataset)
    permutation = torch.randperm(n, generator=g)
    return x[permutation].to(DEVICE), y[permutation].to(DEVICE)


def prepare_data(dataset, seed, include_test=False):
    offset = 100_000 if dataset == "moons" else 0
    split_names = ("train", "validation", "test") if include_test else ("train", "validation")
    data = {}
    for split_id, split in enumerate(split_names):
        data[split] = raw_split(dataset, SPLIT_SIZES[split], offset + seed * 1000 + split_id)
    mean = data["train"][0].mean(dim=0)
    scale = data["train"][0].std(dim=0, unbiased=False)
    assert torch.all(scale > 0)
    for split, (x, y) in data.items():
        data[split] = ((x - mean) / scale, y)
    data["mean"], data["scale"] = mean, scale
    return data


def make_model(dataset, seed):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(seed + (20_000 if dataset == "moons" else 10_000))
        if dataset == "gaussian":
            model = nn.Linear(2, 1)
        else:
            model = nn.Sequential(nn.Linear(2, 8), nn.Tanh(), nn.Linear(8, 1))
    return model.to(device=DEVICE, dtype=DTYPE)


def corruption_uniforms(dataset, seed, n):
    offset = 100_000 if dataset == "moons" else 0
    return torch.rand(n, generator=generator(offset + seed * 1000 + 999), dtype=DTYPE)


def corrupt_labels(clean_labels, uniforms, eta):
    return torch.where(uniforms < eta, 1 - clean_labels, clean_labels)


for dataset, expected in (("gaussian", 3), ("moons", 33)):
    assert sum(p.numel() for p in make_model(dataset, 0).parameters()) == expected


## 3. Fixed objectives and common metrics

The three RobustCE variants use the checked formula from notebook 01:
$$
L(z,y;\delta)=\operatorname{softplus}(z)-yz+
\min(1,\delta)\operatorname{ReLU}((2y-1)z).
$$
Their fixed training radii are 0.05, 0.1, and 0.2. Each variant receives its own validation-selected learning rate.

Smoothing uses
$$
y_\varepsilon=(1-\varepsilon)y+\varepsilon/2,\qquad \varepsilon=0.2.
$$
Observed labels 0 and 1 become targets 0.1 and 0.9.

Evaluate every variant with clean held-out accuracy, ordinary CE, and robust CE at the same fixed radius **delta_eval = 0.1**, regardless of training radius. Learning curves use ordinary clean validation CE.

### Matched-strength comparison: critical analysis (d)

For signed margin $m=(2y-1)z$, label smoothing adds $(\varepsilon/2)m$ to ordinary CE, whereas RobustCE adds $\delta\operatorname{ReLU}(m)$.

Choosing **epsilon = 0.2** matches **RobustCE delta = 0.1** on positive margins, because $\varepsilon/2=\delta=0.1$. Both have individual-example optimal observed-class probability 0.9. This does not force a shared model to have identical predictions or confidence.

For negative margins, RobustCE keeps the nominal CE penalty, while smoothing reduces it. More precisely,
$$
L_{\mathrm{RobustCE}(0.1)}-L_{\mathrm{LS}(0.2)}
=0.1\operatorname{ReLU}(-m).
$$
Thus the comparison aligns the confidence penalty on correctly classified observed examples while retaining the difference on incorrectly classified examples. The losses are not identical. Each method still receives its own learning-rate selection.


In [3]:
def fixed_parameters(method):
    return dict(METHOD_CONFIGS[method])


def loss_per_example(logits, targets, method):
    parameters = fixed_parameters(method)
    if parameters["epsilon"] is not None:
        epsilon = parameters["epsilon"]
        targets = (1 - epsilon) * targets + epsilon / 2
    loss = F.softplus(logits) - targets * logits
    if parameters["delta"] is not None:
        loss = loss + min(1.0, parameters["delta"]) * F.relu((2 * targets - 1) * logits)
    return loss


@torch.no_grad()
def evaluate(model, split):
    x, y = split
    logits = model(x).squeeze(-1)
    if not torch.isfinite(logits).all():
        raise FloatingPointError("Nonfinite prediction.")
    ce = F.binary_cross_entropy_with_logits(logits, y, reduction="none")
    robust = ce + min(1.0, DELTA_EVAL) * F.relu((2 * y - 1) * logits)
    return {
        "ce": ce.mean().item(),
        "accuracy": ((logits >= 0) == y.bool()).to(DTYPE).mean().item(),
        "robust_ce": robust.mean().item(),
        "confidence": torch.sigmoid(logits.abs()).mean().item(),
    }


def observed_training_labels(dataset, seed, data, eta):
    clean = data["train"][1]
    uniforms = corruption_uniforms(dataset, seed, len(clean))
    observed = corrupt_labels(clean, uniforms, eta)
    realized = (observed != clean).to(DTYPE).mean().item()
    return observed, realized


def train_one(dataset, seed, data, observed_labels, method, lr, keep_history=False):
    model = make_model(dataset, seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    x_train = data["train"][0]
    history = []
    started = time.perf_counter()

    def record(epoch):
        model.eval()
        metrics = evaluate(model, data["validation"])
        history.append({"epoch": epoch, "validation_ce": metrics["ce"]})
        model.train()

    if keep_history:
        record(0)
    for epoch in range(1, EPOCHS[dataset] + 1):
        optimizer.zero_grad(set_to_none=True)
        logits = model(x_train).squeeze(-1)
        loss = loss_per_example(logits, observed_labels, method).mean()
        if epoch % HISTORY_EVERY == 0 and not torch.isfinite(loss):
            raise FloatingPointError(f"Nonfinite loss at epoch {epoch}.")
        loss.backward()
        optimizer.step()
        if keep_history and (epoch % HISTORY_EVERY == 0 or epoch == EPOCHS[dataset]):
            record(epoch)

    seconds = time.perf_counter() - started
    model.eval()
    validation = evaluate(model, data["validation"])
    return model, validation, seconds, history


def write_csv(path, records):
    if not records:
        return
    keys = list(dict.fromkeys(key for row in records for key in row))
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=keys)
        writer.writeheader()
        writer.writerows(records)


# Include both margin signs: RobustCE(0.1) and smoothing(0.2) coincide only
# on the positive-margin branch. Their complete loss functions are distinct.
probe_z = torch.tensor([2.0, -2.0])
probe_y = torch.tensor([1.0, 1.0])
probe_losses = [loss_per_example(probe_z, probe_y, method) for method in METHODS]
for first_loss, second_loss in combinations(probe_losses, 2):
    assert not torch.allclose(first_loss, second_loss)


## 4. Tune only the learning rate at eta = 0.1

For each dataset and tuning seed, corrupt the training labels once at $\eta=0.1$. Every variant and learning-rate candidate receives the same observed labels and initial model weights.

Search learning rates 0.001, 0.01, and 0.05 separately for CE, RobustCE(0.05), RobustCE(0.1), RobustCE(0.2), and smoothing with epsilon = 0.2. Select by mean final ordinary CE on clean validation labels over three seeds. No early stopping or test-based selection is used.

There are **90 tuning runs**: 2 datasets x 3 seeds x 5 variants x 3 learning rates. Only learning rate is tuned; delta, epsilon, and corruption probability stay fixed within each variant.


In [4]:
tuning_records = []
failures = []
tuning_started = time.perf_counter()
total_tuning_runs = len(DATASETS) * len(TUNING_SEEDS) * len(METHODS) * len(LEARNING_RATES)
completed = 0

for dataset in DATASETS:
    for seed in TUNING_SEEDS:
        data = prepare_data(dataset, seed, include_test=False)
        observed, realized = observed_training_labels(dataset, seed, data, TUNING_CORRUPTION_RATE)
        for method, lr in product(METHODS, LEARNING_RATES):
            record = dict(
                dataset=dataset, seed=seed, method=method, lr=lr,
                eta=TUNING_CORRUPTION_RATE, realized_corruption=realized,
                **fixed_parameters(method),
            )
            try:
                _, validation, seconds, _ = train_one(
                    dataset, seed, data, observed, method, lr
                )
                record.update(
                    status="ok", validation_ce=validation["ce"],
                    validation_accuracy=validation["accuracy"],
                    validation_robust_ce=validation["robust_ce"], training_seconds=seconds,
                )
            except (FloatingPointError, RuntimeError) as error:
                record.update(status="failed", error=str(error))
                failures.append({"phase": "tuning", **record})
            tuning_records.append(record)
            completed += 1
        write_csv(RESULTS_DIR / "tuning_runs.csv", tuning_records)
        print(f"Tuning: {dataset}, seed {seed} completed ({completed}/{total_tuning_runs}).", flush=True)

print(f"Tuning finished in {time.perf_counter() - tuning_started:.1f} s; failures: {len(failures)}.")


Tuning: gaussian, seed 0 completed (15/90).


Tuning: gaussian, seed 1 completed (30/90).


Tuning: gaussian, seed 2 completed (45/90).


Tuning: moons, seed 0 completed (60/90).


Tuning: moons, seed 1 completed (75/90).


Tuning: moons, seed 2 completed (90/90).


Tuning finished in 33.9 s; failures: 0.


In [5]:
def summarize(values):
    return statistics.mean(values), statistics.stdev(values) if len(values) > 1 else 0.0


tuning_summary = []
for dataset, method, lr in product(DATASETS, METHODS, LEARNING_RATES):
    runs = [
        r for r in tuning_records
        if r["dataset"] == dataset and r["method"] == method and r["lr"] == lr
    ]
    valid = len(runs) == len(TUNING_SEEDS) and all(r["status"] == "ok" for r in runs)
    entry = dict(
        dataset=dataset, method=method, lr=lr, eligible=valid,
        eta=TUNING_CORRUPTION_RATE, **fixed_parameters(method),
    )
    if valid:
        entry["validation_ce_mean"], entry["validation_ce_std"] = summarize(
            [r["validation_ce"] for r in runs]
        )
    tuning_summary.append(entry)

selected = {}
for dataset, method in product(DATASETS, METHODS):
    candidates = [
        r for r in tuning_summary
        if r["dataset"] == dataset and r["method"] == method and r["eligible"]
    ]
    if not candidates:
        raise RuntimeError(f"No complete valid learning-rate candidate for {dataset}/{method}.")
    best = min(candidates, key=lambda r: (r["validation_ce_mean"], r["lr"]))
    selected[(dataset, method)] = {
        "lr": best["lr"], "validation_ce_mean": best["validation_ce_mean"],
        **fixed_parameters(method),
    }
    parameters = fixed_parameters(method)
    parameter = (
        f", fixed delta={parameters['delta']:g}" if parameters["delta"] is not None
        else f", fixed epsilon={parameters['epsilon']:g}" if parameters["epsilon"] is not None else ""
    )
    print(
        f"{dataset} / {method}: lr={best['lr']:g}{parameter}; "
        f"mean clean validation CE={best['validation_ce_mean']:.5f}"
    )
    if best["lr"] in (min(LEARNING_RATES), max(LEARNING_RATES)):
        side = "smallest" if best["lr"] == min(LEARNING_RATES) else "largest"
        print(f"  Selected the {side} tested lr; values outside the grid were not tested.")


def selection_payload():
    return [
        dict(dataset=dataset, method=method, **config)
        for (dataset, method), config in selected.items()
    ]


# Freeze learning rates before generating evaluation datasets.
frozen_settings = selection_payload()
(RESULTS_DIR / "selected_settings.json").write_text(
    json.dumps(frozen_settings, indent=2) + "\n", encoding="utf-8"
)
write_csv(RESULTS_DIR / "tuning_summary.csv", tuning_summary)


gaussian / CE: lr=0.05; mean clean validation CE=0.27815
  Selected the largest tested lr; values outside the grid were not tested.
gaussian / RobustCE(0.05): lr=0.05, fixed delta=0.05; mean clean validation CE=0.32166
  Selected the largest tested lr; values outside the grid were not tested.
gaussian / RobustCE(0.1): lr=0.05, fixed delta=0.1; mean clean validation CE=0.36554
  Selected the largest tested lr; values outside the grid were not tested.
gaussian / RobustCE(0.2): lr=0.05, fixed delta=0.2; mean clean validation CE=0.45560
  Selected the largest tested lr; values outside the grid were not tested.
gaussian / Smoothing(0.2): lr=0.05, fixed epsilon=0.2; mean clean validation CE=0.35297
  Selected the largest tested lr; values outside the grid were not tested.
moons / CE: lr=0.01; mean clean validation CE=0.31794
moons / RobustCE(0.05): lr=0.01, fixed delta=0.05; mean clean validation CE=0.34303
moons / RobustCE(0.1): lr=0.01, fixed delta=0.1; mean clean validation CE=0.37985
moo

### Learning-rate selection

Each point is mean clean validation CE over three tuning seeds, with training labels corrupted at eta = 0.1. Error bars show sample standard deviation. Stars mark each variant's selected learning rate.

All five variants are plotted together. Each RobustCE radius has its own learning-rate selection.


In [6]:
fig = make_subplots(
    rows=1, cols=2, subplot_titles=list(DATASETS), horizontal_spacing=0.12,
)
for col, dataset in enumerate(DATASETS, start=1):
    for method in METHODS:
        points = sorted(
            [r for r in tuning_summary if r["dataset"] == dataset
             and r["method"] == method and r["eligible"]],
            key=lambda r: r["lr"],
        )
        best_lr = selected[(dataset, method)]["lr"]
        fig.add_trace(go.Scatter(
            x=[r["lr"] for r in points],
            y=[r["validation_ce_mean"] for r in points],
            error_y=dict(type="data", array=[r["validation_ce_std"] for r in points]),
            mode="lines+markers", name=method, legendgroup=method,
            showlegend=col == 1, line=dict(color=COLORS[method], dash=LINE_DASHES[method]),
            marker=dict(
                symbol=["star" if r["lr"] == best_lr else "circle" for r in points],
                size=[14 if r["lr"] == best_lr else 7 for r in points],
            ),
            hovertemplate="lr=%{x}<br>validation CE=%{y:.5f}<extra>%{fullData.name}</extra>",
        ), row=1, col=col)
fig.update_xaxes(type="log", title_text="Learning rate", tickvals=list(LEARNING_RATES))
fig.update_yaxes(title_text="Clean validation CE")
fig.update_layout(
    title=f"Learning-rate tuning at eta={TUNING_CORRUPTION_RATE}: five fixed loss variants",
    height=440, margin=dict(t=100, b=80),
    legend=dict(orientation="h", y=-0.25),
)
fig.show(config={"displaylogo": False})


## 5. Head-to-head evaluation at eta = 0 and eta = 0.1

Use five new seeds. For each seed and corruption condition, train every variant **from scratch** with its selected learning rate and fixed loss parameters.

Learning rates were selected only at eta = 0.1 and stay identical across the two evaluation conditions. Data and initial weights match across variants and conditions. All variants share the same corrupted labels at eta = 0.1. Validation and test labels stay clean.

There are **100 evaluation runs**: 2 datasets x 5 seeds x 2 corruption conditions x 5 variants. Realized flip fractions are recorded. All three RobustCE variants appear in every comparison.


In [7]:
evaluation_records = []
representatives = {}
evaluation_started = time.perf_counter()
selected_snapshot = json.dumps(selection_payload(), sort_keys=True)

for dataset in DATASETS:
    for seed in EVALUATION_SEEDS:
        data = prepare_data(dataset, seed, include_test=True)
        for eta in CORRUPTION_RATES:
            observed, realized = observed_training_labels(dataset, seed, data, eta)
            for method in METHODS:
                config = selected[(dataset, method)]
                keep = seed == REPRESENTATIVE_SEED
                record = dict(
                    dataset=dataset, seed=seed, method=method,
                    eta=eta, realized_corruption=realized,
                    lr=config["lr"], delta_eval=DELTA_EVAL, **fixed_parameters(method),
                )
                try:
                    model, validation, seconds, history = train_one(
                        dataset, seed, data, observed, method, config["lr"], keep_history=keep,
                    )
                    test = evaluate(model, data["test"])
                    record.update(
                        status="ok", test_accuracy=test["accuracy"],
                        test_ce=test["ce"], test_robust_ce=test["robust_ce"],
                        test_confidence=test["confidence"],
                        validation_ce=validation["ce"], training_seconds=seconds,
                    )
                    if keep:
                        representatives[(dataset, eta, method)] = {
                            "model": model, "data": data, "history": history,
                        }
                except (FloatingPointError, RuntimeError) as error:
                    record.update(status="failed", error=str(error))
                    failures.append({"phase": "evaluation", **record})
                evaluation_records.append(record)
        write_csv(RESULTS_DIR / "evaluation_runs.csv", evaluation_records)
        print(f"Evaluation: {dataset}, seed {seed} completed.", flush=True)

assert selected_snapshot == json.dumps(selection_payload(), sort_keys=True)
(RESULTS_DIR / "failures.json").write_text(json.dumps(failures, indent=2) + "\n", encoding="utf-8")
print(
    f"Evaluation finished in {time.perf_counter() - evaluation_started:.1f} s. "
    f"Failed runs: {sum(r['status'] == 'failed' for r in evaluation_records)}."
)


Evaluation: gaussian, seed 101 completed.


Evaluation: gaussian, seed 102 completed.


Evaluation: gaussian, seed 103 completed.


Evaluation: gaussian, seed 104 completed.


Evaluation: gaussian, seed 105 completed.


Evaluation: moons, seed 101 completed.


Evaluation: moons, seed 102 completed.


Evaluation: moons, seed 103 completed.


Evaluation: moons, seed 104 completed.


Evaluation: moons, seed 105 completed.


Evaluation finished in 37.7 s. Failed runs: 0.


In [8]:
evaluation_summary = []
for dataset, eta, method in product(DATASETS, CORRUPTION_RATES, METHODS):
    all_runs = [
        r for r in evaluation_records
        if r["dataset"] == dataset and r["eta"] == eta and r["method"] == method
    ]
    runs = [r for r in all_runs if r["status"] == "ok"]
    entry = dict(
        dataset=dataset, eta=eta, method=method,
        successful_runs=len(runs), failed_runs=len(all_runs) - len(runs),
    )
    if runs:
        for metric in ("test_accuracy", "test_ce", "test_robust_ce", "test_confidence", "training_seconds"):
            entry[f"{metric}_mean"], entry[f"{metric}_std"] = summarize([r[metric] for r in runs])
    evaluation_summary.append(entry)
write_csv(RESULTS_DIR / "evaluation_summary.csv", evaluation_summary)

for dataset, method in product(DATASETS, METHODS):
    runs = [r for r in evaluation_records if r["dataset"] == dataset
            and r["method"] == method and r["status"] == "ok"]
    if runs:
        print(
            f"{dataset} / {method}: {len(runs)} successful runs, "
            f"mean training time {statistics.mean(r['training_seconds'] for r in runs):.3f} s."
        )
    else:
        print(f"{dataset} / {method}: all evaluation runs failed.")
print(f"Detailed metrics saved to {RESULTS_DIR}; no tables displayed.")


gaussian / CE: 10 successful runs, mean training time 0.167 s.
gaussian / RobustCE(0.05): 10 successful runs, mean training time 0.192 s.
gaussian / RobustCE(0.1): 10 successful runs, mean training time 0.202 s.
gaussian / RobustCE(0.2): 10 successful runs, mean training time 0.195 s.
gaussian / Smoothing(0.2): 10 successful runs, mean training time 0.172 s.
moons / CE: 10 successful runs, mean training time 0.527 s.
moons / RobustCE(0.05): 10 successful runs, mean training time 0.579 s.
moons / RobustCE(0.1): 10 successful runs, mean training time 0.582 s.
moons / RobustCE(0.2): 10 successful runs, mean training time 0.572 s.
moons / Smoothing(0.2): 10 successful runs, mean training time 0.528 s.
Detailed metrics saved to C:\Users\shich\Src\hw1opt\results\02_fixed_losses; no tables displayed.


### Head-to-head clean test metrics

Compare CE, RobustCE(0.05), RobustCE(0.1), RobustCE(0.2), and smoothing at both corruption levels using their frozen learning rates. Error bars show sample standard deviation over five matched seeds, varying datasets and initialization.

Ordinary CE, common-radius robust CE, and accuracy are distinct criteria. Robust test CE always uses delta_eval = 0.1, regardless of training delta. Different datasets do not directly rank architectures.


In [9]:
metric_specs = (
    ("test_accuracy", "Clean test accuracy (%)", 100.0),
    ("test_ce", "Ordinary clean test CE", 1.0),
    ("test_robust_ce", "Common-radius robust test CE", 1.0),
)
fig = make_subplots(
    rows=2, cols=3, vertical_spacing=0.20,
    subplot_titles=[f"{dataset}<br>{title}" for dataset in DATASETS for _, title, _ in metric_specs],
)
for row, dataset in enumerate(DATASETS, start=1):
    for col, (metric, _, scale) in enumerate(metric_specs, start=1):
        for method in METHODS:
            points = [
                r for r in evaluation_summary
                if r["dataset"] == dataset and r["method"] == method and r["successful_runs"] > 0
            ]
            fig.add_trace(go.Scatter(
                x=[r["eta"] for r in points],
                y=[scale * r[f"{metric}_mean"] for r in points],
                error_y=dict(type="data", array=[scale * r[f"{metric}_std"] for r in points]),
                mode="lines+markers", name=method, legendgroup=method,
                showlegend=row == 1 and col == 1,
                line=dict(color=COLORS[method], dash=LINE_DASHES[method]),
                hovertemplate="eta=%{x:.1f}<br>mean=%{y:.5f}<extra>%{fullData.name}</extra>",
            ), row=row, col=col)
fig.update_xaxes(
    title_text="Training-label flip probability eta",
    tickvals=list(CORRUPTION_RATES), range=[-0.01, 0.11],
)
fig.update_layout(
    title=f"Head-to-head: five variants, lr tuned at eta={TUNING_CORRUPTION_RATE}",
    height=780, margin=dict(t=120, b=90),
    legend=dict(orientation="h", y=-0.14),
)
fig.show(config={"displaylogo": False})


### Learning curves at a common metric

Representative plots use seed **101** at eta = 0 and eta = 0.1. All curves show ordinary CE on clean validation labels. The same selected learning rate is used at both corruption levels for each method and dataset.


In [10]:
fig = make_subplots(
    rows=2, cols=2, vertical_spacing=0.18,
    subplot_titles=[
        f"{dataset}: eta={eta:g}, seed={REPRESENTATIVE_SEED}"
        for dataset in DATASETS for eta in PLOT_CORRUPTION_RATES
    ],
)
for row, dataset in enumerate(DATASETS, start=1):
    for col, eta in enumerate(PLOT_CORRUPTION_RATES, start=1):
        for method in METHODS:
            representative = representatives.get((dataset, eta, method))
            if representative is None:
                continue
            history = representative["history"]
            fig.add_trace(go.Scatter(
                x=[r["epoch"] for r in history], y=[r["validation_ce"] for r in history],
                mode="lines", name=method, legendgroup=method,
                showlegend=row == 1 and col == 1, line=dict(color=COLORS[method], dash=LINE_DASHES[method]),
            ), row=row, col=col)
fig.update_xaxes(title_text="Epoch")
fig.update_yaxes(title_text="Ordinary clean validation CE")
fig.update_layout(
    title="Learning progress under the same validation criterion",
    height=740, margin=dict(t=100, b=90), legend=dict(orientation="h", y=-0.14),
)
fig.show(config={"displaylogo": False})


### Confidence and radius: critical analysis (b)

Compare all five variants using mean clean-test confidence, $\operatorname{mean}\max(p,1-p)$. Error bars show sample standard deviation across five matched seeds. This statistic measures confidence, not calibration.

For one example, the minimizing positive margin is $m^*=\log((1-\delta)/\delta)$. Delta = 0.05 corresponds to observed-class probability 0.95; delta = 0.1 corresponds to 0.9; delta = 0.2 corresponds to 0.8. These individual-example optima are not hard caps on a shared model's predictions.

The paired changes below compare every pair of RobustCE radii on the same seed and corruption condition. RobustCE(0.1) and smoothing(0.2) have the same additional penalty on positive margins, but differ on negative margins.


In [11]:
fig = make_subplots(rows=1, cols=2, subplot_titles=list(DATASETS), horizontal_spacing=0.12)
for col, dataset in enumerate(DATASETS, start=1):
    for method in METHODS:
        points = [r for r in evaluation_summary if r["dataset"] == dataset and r["method"] == method and r["successful_runs"] > 0]
        fig.add_trace(go.Scatter(
            x=[r["eta"] for r in points],
            y=[100 * r["test_confidence_mean"] for r in points],
            error_y=dict(type="data", array=[100 * r["test_confidence_std"] for r in points]),
            mode="lines+markers", name=method, legendgroup=method, showlegend=col == 1,
            line=dict(color=COLORS[method], dash=LINE_DASHES[method]),
            hovertemplate="eta=%{x:.1f}<br>mean confidence=%{y:.2f}%<extra>%{fullData.name}</extra>",
        ), row=1, col=col)
fig.update_xaxes(title_text="Training-label flip probability eta", tickvals=list(CORRUPTION_RATES), range=[-0.01, 0.11])
fig.update_yaxes(title_text="Mean clean-test confidence (%)", range=[50, 100])
fig.update_layout(
    title="Confidence across five fixed loss variants",
    height=470, margin=dict(t=90, b=90), legend=dict(orientation="h", y=-0.25),
)
fig.show(config={"displaylogo": False})

radius_changes = []
robust_methods = sorted(
    (method for method in METHODS if METHOD_CONFIGS[method]["delta"] is not None),
    key=lambda method: METHOD_CONFIGS[method]["delta"],
)
for dataset, eta in product(DATASETS, CORRUPTION_RATES):
    paired = {
        method: {
            r["seed"]: r for r in evaluation_records
            if r["dataset"] == dataset and r["eta"] == eta and r["method"] == method and r["status"] == "ok"
        }
        for method in robust_methods
    }
    for method_from, method_to in combinations(robust_methods, 2):
        seeds = sorted(set(paired[method_from]) & set(paired[method_to]))
        if not seeds:
            continue
        delta_from = METHOD_CONFIGS[method_from]["delta"]
        delta_to = METHOD_CONFIGS[method_to]["delta"]
        changes = [
            100 * (paired[method_to][seed]["test_confidence"] - paired[method_from][seed]["test_confidence"])
            for seed in seeds
        ]
        mean_change, std_change = summarize(changes)
        radius_changes.append(dict(
            dataset=dataset, eta=eta, method_from=method_from, method_to=method_to,
            delta_from=delta_from, delta_to=delta_to, paired_seeds=len(seeds),
            confidence_change_percentage_points_mean=mean_change,
            confidence_change_percentage_points_std=std_change,
        ))
        print(
            f"{dataset}, eta={eta:g}: confidence change for delta {delta_from:g} -> {delta_to:g} is "
            f"{mean_change:+.2f} +/- {std_change:.2f} percentage points "
            f"(sample SD, {len(seeds)} matched seeds)."
        )
write_csv(RESULTS_DIR / "radius_confidence_changes.csv", radius_changes)


gaussian, eta=0: confidence change for delta 0.05 -> 0.1 is -5.15 +/- 0.13 percentage points (sample SD, 5 matched seeds).
gaussian, eta=0: confidence change for delta 0.05 -> 0.2 is -13.97 +/- 0.25 percentage points (sample SD, 5 matched seeds).
gaussian, eta=0: confidence change for delta 0.1 -> 0.2 is -8.81 +/- 0.13 percentage points (sample SD, 5 matched seeds).
gaussian, eta=0.1: confidence change for delta 0.05 -> 0.1 is -3.84 +/- 0.09 percentage points (sample SD, 5 matched seeds).
gaussian, eta=0.1: confidence change for delta 0.05 -> 0.2 is -10.98 +/- 0.22 percentage points (sample SD, 5 matched seeds).
gaussian, eta=0.1: confidence change for delta 0.1 -> 0.2 is -7.14 +/- 0.13 percentage points (sample SD, 5 matched seeds).
moons, eta=0: confidence change for delta 0.05 -> 0.1 is -4.16 +/- 0.37 percentage points (sample SD, 5 matched seeds).
moons, eta=0: confidence change for delta 0.05 -> 0.2 is -12.45 +/- 0.39 percentage points (sample SD, 5 matched seeds).
moons, eta=0: c

### Decision boundaries and confidence

Each corruption condition shows all five variants on both datasets, with one shared probability scale across all ten panels. The black contour is the 0.5 decision boundary. Markers show 200 clean test examples (circle: class 0; cross: class 1). Coordinates use training-feature standardization.

Compare RobustCE(0.05), RobustCE(0.1), and RobustCE(0.2) to distinguish changes in probability saturation from movement of the decision boundary. Softer predictions alone do not establish better calibration.


In [12]:
for eta in PLOT_CORRUPTION_RATES:
    fig = make_subplots(
        rows=2, cols=len(METHODS), vertical_spacing=0.22, horizontal_spacing=0.04,
        subplot_titles=[f"{dataset}<br>{method}" for dataset in DATASETS for method in METHODS],
    )
    for row, dataset in enumerate(DATASETS, start=1):
        reference = next(
            (representatives[(dataset, eta, method)] for method in METHODS
             if (dataset, eta, method) in representatives), None
        )
        if reference is None:
            continue
        data = reference["data"]
        x_train = data["train"][0]
        lower, upper = x_train.min(dim=0).values - 0.25, x_train.max(dim=0).values + 0.25
        gx = torch.linspace(lower[0], upper[0], 101)
        gy = torch.linspace(lower[1], upper[1], 101)
        yy, xx = torch.meshgrid(gy, gx, indexing="ij")
        grid = torch.stack((xx.flatten(), yy.flatten()), dim=1)
        x_test, y_test = data["test"]
        for col, method in enumerate(METHODS, start=1):
            representative = representatives.get((dataset, eta, method))
            if representative is None:
                continue
            with torch.no_grad():
                probability = torch.sigmoid(representative["model"](grid).squeeze(-1)).reshape(101, 101)
            fig.add_trace(go.Contour(
                x=gx.tolist(), y=gy.tolist(), z=probability.tolist(),
                coloraxis="coloraxis", contours=dict(coloring="heatmap", showlines=False),
                hovertemplate="x1=%{x:.2f}<br>x2=%{y:.2f}<br>p(class 1)=%{z:.3f}<extra></extra>",
                showscale=False, showlegend=False,
            ), row=row, col=col)
            fig.add_trace(go.Contour(
                x=gx.tolist(), y=gy.tolist(), z=probability.tolist(),
                contours=dict(start=0.5, end=0.5, size=1, coloring="none"),
                line=dict(color="#111827", width=2),
                showscale=False, showlegend=False, hoverinfo="skip",
            ), row=row, col=col)
            for label, symbol in ((0, "circle"), (1, "cross")):
                mask = y_test[:200] == label
                fig.add_trace(go.Scatter(
                    x=x_test[:200, 0][mask].tolist(), y=x_test[:200, 1][mask].tolist(),
                    mode="markers", name=f"Class {label}", legendgroup=f"class-{label}",
                    showlegend=row == 1 and col == 1,
                    marker=dict(
                        symbol=symbol, size=5, opacity=0.65,
                        color="#111827" if label == 0 else "#ffffff",
                        line=dict(color="#111827", width=0.6),
                    ),
                ), row=row, col=col)
            fig.update_xaxes(range=[lower[0].item(), upper[0].item()], row=row, col=col)
            fig.update_yaxes(range=[lower[1].item(), upper[1].item()], row=row, col=col)
    fig.update_xaxes(title_text="Standardized x1", row=2)
    fig.update_yaxes(title_text="Standardized x2", col=1)
    fig.update_layout(
        title=f"Probabilities and boundaries: eta={eta:g}, seed={REPRESENTATIVE_SEED}",
        height=780, margin=dict(t=100, b=80),
        coloraxis=dict(cmin=0, cmax=1, colorscale="RdBu_r", colorbar=dict(title="p(class 1)")),
        legend=dict(orientation="h", y=-0.12),
    )
    fig.show(config={"displaylogo": False})


## 6. Reproducibility and interpretation

Saved records include each variant's delta or epsilon, selected learning rate, seed, corruption probability, realized flip fraction, clean held-out metrics, and training time. Representative models and learning histories are saved.

Compare variants within a corruption condition, then compare how each changes between eta = 0 and eta = 0.1. All three RobustCE radii are fixed comparison variants; none is selected using test data. Only learning rates are selected at eta = 0.1 using clean validation CE.

This experiment uses two datasets, three training radii, two corruption conditions, three tuning seeds, finite learning-rate grids, and fixed epoch budgets. The observed confidence change does not establish monotonicity for every radius or architecture. Better common-radius robust CE alone does not establish better accuracy or calibration.


In [13]:
protocol = {
    "torch_version": str(torch.__version__),
    "device": str(DEVICE), "dtype": str(DTYPE), "num_threads": torch.get_num_threads(),
    "datasets": list(DATASETS), "methods": list(METHODS),
    "method_parameters": METHOD_CONFIGS,
    "learning_rates": list(LEARNING_RATES), "training_deltas": sorted(p["delta"] for p in METHOD_CONFIGS.values() if p["delta"] is not None), "epsilon": EPSILON,
    "tuning_corruption_rate": TUNING_CORRUPTION_RATE,
    "evaluation_corruption_rates": list(CORRUPTION_RATES), "delta_eval": DELTA_EVAL,
    "tuning_seeds": list(TUNING_SEEDS), "evaluation_seeds": list(EVALUATION_SEEDS),
    "split_sizes": SPLIT_SIZES, "epochs": EPOCHS,
    "optimizer": "Adam, default betas/eps, no weight decay, no schedule",
    "batching": "full batch", "tuned_parameters": ["lr"],
    "smoothing_matched_to_robust_delta": 0.1,
    "selection": "mean final clean validation CE after training at eta=0.1, separately per variant",
    "error_bars": "sample standard deviation across matched seeds",
    "representative_seed": REPRESENTATIVE_SEED,
    "failed_tuning_runs": sum(r["status"] == "failed" for r in tuning_records),
    "failed_evaluation_runs": sum(r["status"] == "failed" for r in evaluation_records),
}
(RESULTS_DIR / "protocol.json").write_text(json.dumps(protocol, indent=2) + "\n", encoding="utf-8")
histories = [
    {"dataset": dataset, "eta": eta, "method": method, "history": item["history"]}
    for (dataset, eta, method), item in representatives.items()
]
(RESULTS_DIR / "representative_histories.json").write_text(
    json.dumps(histories, indent=2) + "\n", encoding="utf-8"
)
torch.save(
    {
        f"{dataset}/eta={eta:g}/{method}": {
            "state_dict": item["model"].state_dict(),
            "seed": REPRESENTATIVE_SEED, "dataset": dataset, "eta": eta,
            "method": method, **selected[(dataset, method)],
            "feature_mean": item["data"]["mean"], "feature_scale": item["data"]["scale"],
        }
        for (dataset, eta, method), item in representatives.items()
    },
    RESULTS_DIR / "representative_models.pt",
)

assert len(tuning_records) == total_tuning_runs
assert len(evaluation_records) == len(DATASETS) * len(EVALUATION_SEEDS) * len(CORRUPTION_RATES) * len(METHODS)
assert all(r["eta"] == TUNING_CORRUPTION_RATE for r in tuning_records)
assert all(r["delta_eval"] == DELTA_EVAL for r in evaluation_records)
for dataset, seed, eta in product(DATASETS, EVALUATION_SEEDS, CORRUPTION_RATES):
    matched = [r for r in evaluation_records if r["dataset"] == dataset and r["seed"] == seed and r["eta"] == eta]
    assert len({r["realized_corruption"] for r in matched}) == 1
    assert {r["method"] for r in matched} == set(METHODS)
for record in tuning_records + evaluation_records:
    assert record["delta"] == METHOD_CONFIGS[record["method"]]["delta"]
    assert record["epsilon"] == METHOD_CONFIGS[record["method"]]["epsilon"]
print(
    f"Complete: {len(tuning_records)} tuning runs and {len(evaluation_records)} evaluation runs. "
    f"Recorded failures: {len(failures)}."
)
print(f"Artifacts: {RESULTS_DIR}")


Complete: 90 tuning runs and 100 evaluation runs. Recorded failures: 0.
Artifacts: C:\Users\shich\Src\hw1opt\results\02_fixed_losses
